# 🚗 Driver Guardian AI — End-to-End DrowsyClassifier Training Pipeline (Kaggle Edition)
## Huấn luyện Mô hình Phát hiện Ngủ gật Toàn diện Trực tiếp từ Video Thô qua Backbone PAFPN & SpatioTemporal ConvGRU

> **Đặc tả Kỹ thuật:**
> - **Kiến trúc mô hình**: `DrowsyClassifier` (End-to-End SpatioTemporal Deep Learning). Kết hợp trực tiếp Spatial Backbone, PAFPN Feature Aggregator và Temporal ConvGRU với Dual Attention.
> - **Huấn luyện từ đầu (Train from Scratch)**: Không phụ thuộc vào nạp trọng số Detector bên ngoài; gradient lan truyền trực tiếp xuyên suốt từ ConvGRU về Backbone.
> - **Tối ưu hóa VRAM**: Kích hoạt Gradient Checkpointing cho Backbone/PAFPN (giảm 50-60% VRAM khi lan truyền ngược) và tùy chọn Freeze BN.
> - **Tăng cường chuỗi video**: `VideoAugmenter` bảo toàn tính nhất quán thời gian (Temporal Consistency) qua `apply_sequence` đồng bộ 1 random seed duy nhất cho mọi khung hình.
> - **Dữ liệu**: Nạp video thô (`.mp4`, `.avi`, `.mkv`), lấy mẫu đều đặn (`sample_interval=0.2s`), tensor CPU `uint8` siêu nhẹ chống tràn RAM.
> - **Tối ưu hóa**: AdamW / SGD, Cosine Annealing LR, Mixed Precision (AMP FP16), Gradient Accumulation (boundary-safe), Early Stopping, CUDA OOM Recovery.
> - **Cơ chế Khôi phục (Resume)**: Hỗ trợ tiếp tục train từ checkpoint bất kỳ (`last.pt`, `best.pt`, `epoch_N.pt`) kèm khôi phục đầy đủ GradScaler, Optimizer, Scheduler.
> - **Môi trường**: Kaggle Notebook GPU Kernel (Tesla T4 / P100 16GB VRAM, Ubuntu Linux) hoặc Google Colab / Local Windows.

---
### 💡 Hướng dẫn Cấu hình trên Kaggle:
1. **Bật GPU**: Menu bên phải `Settings` $\rightarrow$ `Accelerator` $\rightarrow$ Chọn **GPU T4 x1** hoặc **GPU P100**.
2. **Thêm Dữ liệu (Add Data)**:
   - Thêm Dataset video ULDD đã qua tiền xử lý (chứa `dataset_merged_split.csv` và các video clip).
3. **Cập nhật đường dẫn tại Cell 5**: Kiểm tra và trỏ `dataset_dir` tới đúng thư mục trong `/kaggle/input/...` hoặc thư mục tải từ `kagglehub`.

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES (IF NEEDED).
import kagglehub
kagglehub.login()

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL OR CUSTOMIZE PATHS IN CELL 5.
try:
    # nyvantran6634_ul_dd4ni_path = kagglehub.dataset_download('nyvantran6634/ul-dd4ni', output_dir="./kagglehub/nyvantran6634/ul-dd4ni")
    # print(f"[✓] nyvantran6634_ul_dd4ni_path: {nyvantran6634_ul_dd4ni_path}")
    nyvantran6634_dataset_datn4ni2_path = kagglehub.dataset_download('nyvantran6634/dataset-datn4ni2', output_dir="./kagglehub/nyvantran6634/dataset-datn4ni2")
    print(f"nyvantran6634_dataset_datn4ni2_path: {nyvantran6634_dataset_datn4ni2_path}")
except Exception as e:
    print(f"[*] Bỏ qua download kagglehub hoặc dataset đã có sẵn cục bộ: {e}")

In [ ]:
# 1. Cài đặt các thư viện bổ sung nếu cần và import dependencies
%load_ext tensorboard
import os
import sys
import time
import math
import random
import csv
import json
import shutil
import zipfile
import inspect
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import List, Tuple, Dict, Any, Optional, Union, Sequence

import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.checkpoint
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

# Import Albumentations (tự động cài đặt nếu thiếu)
try:
    import albumentations as A
except ImportError:
    import subprocess
    print("[*] Đang cài đặt thư viện albumentations...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "albumentations"])
    import albumentations as A

# Thiết lập hiển thị đồ thị inline
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print(f"[✓] PyTorch Version      : {torch.__version__}")
print(f"[✓] OpenCV Version       : {cv2.__version__}")
print(f"[✓] Albumentations       : {A.__version__}")
print(f"[✓] CUDA Sẵn sàng        : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[✓] Thiết bị GPU         : {torch.cuda.get_device_name(0)}")
    print(f"[✓] Số lượng GPU         : {torch.cuda.device_count()}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"[✓] Tổng VRAM            : {vram_gb:.2f} GB")
else:
    print("[!] CẢNH BÁO: Không phát hiện GPU. Vui lòng bật GPU Accelerator (T4 x1 hoặc P100) trong menu Kaggle Settings!")

In [ ]:
# 2. Cố định Seed ngẫu nhiên để đảm bảo tính tái lập (Reproducibility)
def seed_everything(seed: int = 42) -> None:
    """Cố định seed cho tất cả các thư viện để kết quả huấn luyện có thể tái lập hoàn toàn."""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"[✓] Đã cố định toàn bộ hạt giống ngẫu nhiên: SEED = {seed}")


seed_everything(42)

## ⚙️ Section 2: Cấu hình Hệ thống Tập trung Duy nhất (`TrainConfig`)
Toàn bộ siêu tham số của mô hình, dữ liệu, hàm loss, bộ tối ưu hóa và logging được bao phủ đầy đủ 100% từ 7 nhóm trong `configs/config.yaml`.
Vui lòng kiểm tra và tùy chỉnh các đường dẫn `/kaggle/input/...` cho phù hợp với môi trường của bạn.

In [ ]:
# CELL CẤU HÌNH TẬP TRUNG DUY NHẤT (SINGLE CONFIGURATION CELL)
@dataclass
class TrainConfig:
    """
    Cấu hình toàn diện cho pipeline huấn luyện End-to-End DrowsyClassifier.
    Đồng bộ đầy đủ 100% tất cả các thông số từ 7 nhóm trong configs/config.yaml.
    """
    # =========================================================================
    # 1. DATASET CONFIGURATION (RAW VIDEO)
    # =========================================================================
    # Thư mục gốc chứa video thô (đã quét nhãn hoặc có cấu trúc train/val)
    dataset_dir: str = r"./kagglehub/nyvantran6634/dataset-datn4ni2/data_processed"
    # Tệp manifest CSV chứa danh sách video clip, nhãn và trường phân chia split
    manifest_file: Optional[str] = "dataset_merged_split.csv"
    # [Tham chiếu train.py] Trọng số BackboneNeck NMSFreeDetector (không nạp khi train DrowsyClassifier from scratch)
    backbone_neck_checkpoint: Optional[str] = None
    # Khoảng thời gian (giây) giữa các khung hình lấy mẫu (FPS hiệu dụng = 5.0)
    sample_interval: float = 0.2
    # Số lượng khung hình cố định (None = dynamic padding theo độ dài clip)
    seq_len: Optional[int] = None
    # Kích thước khung hình (Height, Width) đưa vào Backbone
    image_size: Tuple[int, int] = (224, 224)
    # Các định dạng video hợp lệ được hỗ trợ
    video_exts: Tuple[str, ...] = (".avi", ".mp4", ".mkv", ".mov")
    # Số khung hình tối thiểu để coi là một video hợp lệ
    min_frames: int = 10
    # Bật/tắt tăng cường dữ liệu chuỗi video đồng bộ thời gian
    use_augmentation: bool = True

    # =========================================================================
    # 2. DATALOADER CONFIGURATION
    # =========================================================================
    # Kích thước batch size cho tập train và val (nhỏ hơn để vừa VRAM khi train End-to-End)
    batch_size: int = 32
    val_batch_size: int = 16
    # Số tiến trình worker tải dữ liệu (Linux Kaggle đặt 2, Windows đặt 0)
    num_workers: int = 2
    val_num_workers: int = 2
    # Khóa vùng nhớ GPU (False để tránh tràn pinned host memory)
    pin_memory: bool = False
    # Trộn ngẫu nhiên tập huấn luyện
    shuffle: bool = True
    # Hạt giống ngẫu nhiên để tái lập kết quả
    seed: int = 42
    # [DÙNG CHUNG] Kích thước mini-chunk chia nhỏ khung hình khi qua Backbone/PAFPN (khống chế đỉnh VRAM O(chunk_size))
    chunk_size: int = 32

    # =========================================================================
    # 3. MODEL ARCHITECTURE CONFIGURATION
    # =========================================================================
    # Số lượng lớp phân loại đầu ra (2: Tỉnh táo / Buồn ngủ)
    num_classes: int = 2
    # Số kênh đặc trưng sau cổ giảm kênh SpatialReductionNeck (nén 448 -> 128)
    input_dim: int = 128
    convgru_in_dim: int = 128
    # Số kênh ẩn của từng tầng ConvGRU
    hidden_dim: Union[int, Tuple[int, ...]] = 128
    convgru_hidden_dim: Union[int, Tuple[int, ...]] = 128
    # Số tầng ConvGRU xếp chồng
    num_layers: int = 2
    convgru_num_layers: int = 2
    # Tỷ lệ Dropout cho đầu phân loại FC Head
    dropout: float = 0.35
    # Cơ chế giám sát chuỗi thời gian ('attention_pooling' | 'sequence')
    supervision_mode: str = "attention_pooling"
    # Kích thước lưới không gian hội tụ (Height, Width) sau SpatialReductionNeck
    target_size: Tuple[int, int] = (14, 14)

    # --- THAM SỐ DÀNH RIÊNG CHO DROWSYCLASSIFIER (src/models1.py & src/train1.py) ---
    # Độ rộng kênh các tầng Backbone (c1, c2, c3, c4, c5)
    backbone_w: Tuple[int, ...] = (16, 32, 64, 128, 256)
    # Số lượng khối lặp lại trong từng tầng Backbone
    backbone_n: Tuple[int, ...] = (1, 2, 2, 1)
    # Số lượng khối lặp lại trong PAFPN
    neck_n: int = 1
    # Bật gradient checkpointing cho Backbone/Neck tiết kiệm VRAM
    gradient_checkpointing: bool = True
    # Khóa running mean/var của BatchNorm trong Backbone/Neck (triệt tiêu double-update và bảo vệ phân phối khi padding)
    freeze_bn: bool = True

    # --- THAM SỐ DÀNH RIÊNG CHO CONVGRU TÁCH RỜI (src/models.py & src/train.py) ---
    # Kênh đặc trưng của các tầng (p3, p4, p5) từ NMSFreeDetector PAFPN
    cnn_neck_channels: Tuple[int, int, int] = (64, 128, 256)
    # Tỷ lệ Dropout sau cổ giảm kênh SpatialReductionNeck
    adapter_dropout: float = 0.25

    # =========================================================================
    # 4. LOSS CONFIGURATION
    # =========================================================================
    # Loại hàm mất mát: "ce" (CrossEntropyLoss) hoặc "bce" (BCEWithLogitsLoss)
    loss_type: str = "ce"
    # Hằng số epsilon kẹp giá trị số học khi dùng BCE
    bce_eps: float = 1e-7
    # Cách gom nhóm loss ('mean' | 'sum')
    bce_reduction: str = "mean"
    # Trọng số cho lớp dương (1: Buồn ngủ) khi dữ liệu mất cân bằng
    pos_weight: Optional[float] = None

    # =========================================================================
    # 5. OPTIMIZER & SCHEDULER CONFIGURATION
    # =========================================================================
    # Tổng số epoch huấn luyện
    epochs: int = 20
    # Learning rate khởi tạo
    lr0: float = 0.001
    # Tỷ lệ lr tối thiểu: lr_min = lr0 * lr_min_factor (1e-5)
    lr_min_factor: float = 0.01
    # Hệ số phạt L2 Regularization (Weight Decay)
    weight_decay: float = 0.0001
    # Bộ tối ưu hóa: "adamw" | "adam" | "sgd"
    optimizer: str = "adamw"
    # Tham số betas cho Adam/AdamW
    betas: Tuple[float, float] = (0.9, 0.999)
    # Động lượng Momentum khi optimizer="sgd"
    momentum: float = 0.9
    # Ngưỡng cắt gradient (Gradient Clipping)
    grad_clip_norm: float = 1.0
    # Số bước tích lũy gradient trước khi cập nhật trọng số
    gradient_accumulation_steps: int = 1
    # Khoảng cách epoch giữa các lần dọn rác GPU bộ nhớ đệm
    empty_cache_interval: int = 2
    # Bật/tắt bộ điều chỉnh Learning Rate Scheduler (CosineAnnealingLR)
    use_scheduler: bool = True

    # =========================================================================
    # 6. CHECKPOINTS, EARLY STOPPING & LOGGING CONFIGURATION
    # =========================================================================
    # Thư mục lưu log TensorBoard
    tb_log_dir: str = "./logs/tensorboard"
    # Tên bài thử nghiệm huấn luyện
    experiment_name: str = "drowsy_classifier_end2end"
    # Thư mục lưu checkpoint mô hình (.pt)
    checkpoint_dir: str = "./checkpoints/drowsy_end2end"
    # Lưu checkpoint cho tất cả các epoch
    save_all_epochs: bool = True
    # Khoảng epoch giữa 2 lần lưu định kỳ
    save_ckpt_interval_epochs: int = 1
    # Số lượng checkpoint định kỳ tối đa giữ lại (None = giữ toàn bộ)
    ckpt_keep_last: Optional[int] = None
    # Bật/tắt tiếp tục huấn luyện từ checkpoint
    enable_resume: bool = False
    # Đường dẫn file checkpoint hoặc bí danh ('last', 'best') để resume
    resume: str = ""
    # Cấu hình Giám sát Phân giải cao theo từng Step (Per-Step Metrics)
    enable_step_logging: bool = True
    log_step_interval: int = 5
    # Cơ chế Dừng sớm (Early Stopping)
    early_stopping: bool = True
    patience: int = 10
    monitor_metric: str = "val_f1"
    monitor_mode: str = "max"
    min_delta: float = 0.0001
    # File lưu bảng lịch sử các epoch huấn luyện
    history_csv_path: str = "./logs/training_history.csv"

    # =========================================================================
    # 7. RUNTIME & HARDWARE CONFIGURATION
    # =========================================================================
    # Thiết bị tính toán ("cuda" | "cpu")
    device: str = "cuda"
    # Bật Tự động ép kiểu hỗn hợp FP16 (Automatic Mixed Precision)
    amp: bool = True 
    # Số epoch giữa 2 lần đánh giá tập Validation
    val_interval_epochs: int = 1
    # Bật thanh tiến trình giám sát trực quan tqdm
    use_tqdm: bool = True

    def __post_init__(self):
        # Đồng bộ các bí danh đọc bởi ConvGRU
        if self.convgru_hidden_dim != self.hidden_dim:
            self.convgru_hidden_dim = self.hidden_dim
        if self.convgru_in_dim != self.input_dim:
            self.convgru_in_dim = self.input_dim
        if self.convgru_num_layers != self.num_layers:
            self.convgru_num_layers = self.num_layers

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


cfg = TrainConfig()
print("[✓] Đã khởi tạo cấu hình TrainConfig thành công!")
print(f"[*] Thiết bị mục tiêu : {cfg.device}")
print(f"[*] Batch size        : {cfg.batch_size} (Accumulation Steps: {cfg.gradient_accumulation_steps} -> Effective: {cfg.batch_size * cfg.gradient_accumulation_steps})")
print(f"[*] Mini-Chunk Size   : {cfg.chunk_size}")
print(f"[*] Gradient Checkpoint : {cfg.gradient_checkpointing} | Freeze BN: {cfg.freeze_bn}")
print(f"[*] Tổng Epochs       : {cfg.epochs}")

## 🧠 Section 3: Định nghĩa Kiến trúc Backbone PAFPN
Định nghĩa đầy đủ các khối CNN cơ sở (`Conv`, `Bottleneck`, `C2f`, `CIB`, `C2fCIB`, `SPPF`, `C2fPSA`, `SCDown`) và các module kết nối `Backbone`, `PAFPN`.

In [ ]:
# 1. Các khối tích chập chuẩn hóa của NMSFreeDetector PAFPN
def autopad(k, p=None, d=1):
    """Tự động tính toán padding để bảo toàn kích thước không gian."""
    if p is not None:
        return p
    return (d * (k - 1) + 1) // 2


class Conv(nn.Module):
    def __init__(self, c1, c2, k=1, s=1, p=None, g=1, d=1, act=True):
        super().__init__()
        self.conv = nn.Conv2d(c1, c2, k, s, autopad(k, p, d), groups=g, dilation=d, bias=False)
        self.bn = nn.BatchNorm2d(c2)
        self.act = nn.SiLU(inplace=True) if act else nn.Identity()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class Bottleneck(nn.Module):
    def __init__(self, c1, c2, shortcut=True, e=0.5):
        super().__init__()
        c_ = int(c2 * e)
        self.cv1 = Conv(c1, c_, 3, 1)
        self.cv2 = Conv(c_, c2, 3, 1)
        self.add = shortcut and c1 == c2

    def forward(self, x):
        y = self.cv2(self.cv1(x))
        return x + y if self.add else y


class C2f(nn.Module):
    def __init__(self, c1, c2, n=1, shortcut=True, e=0.5):
        super().__init__()
        self.c = int(c2 * e)
        self.cv1 = Conv(c1, 2 * self.c, 1, 1)
        self.cv2 = Conv((2 + n) * self.c, c2, 1, 1)
        self.m = nn.ModuleList(Bottleneck(self.c, self.c, shortcut, e=1.0) for _ in range(n))

    def forward(self, x):
        y = list(self.cv1(x).chunk(2, 1))
        for m in self.m:
            y.append(m(y[-1]))
        return self.cv2(torch.cat(y, 1))


class CIB(nn.Module):
    def __init__(self, c1, c2, shortcut=True, e=0.5, large_kernel=False):
        super().__init__()
        c_ = int(c2 * e)
        k = 7 if large_kernel else 3
        self.block = nn.Sequential(
            Conv(c1, c1, k, 1, g=c1),
            Conv(c1, 2 * c_, 1, 1),
            Conv(2 * c_, 2 * c_, k, 1, g=2 * c_),
            Conv(2 * c_, c2, 1, 1),
            Conv(c2, c2, k, 1, g=c2),
        )
        self.add = shortcut and c1 == c2

    def forward(self, x):
        y = self.block(x)
        return x + y if self.add else y


class C2fCIB(C2f):
    def __init__(self, c1, c2, n=1, shortcut=False, e=0.5, large_kernel=False):
        super().__init__(c1, c2, n, shortcut, e)
        self.m = nn.ModuleList(CIB(self.c, self.c, shortcut, e=1.0, large_kernel=large_kernel) for _ in range(n))


class SPPF(nn.Module):
    def __init__(self, c1, c2, k=5):
        super().__init__()
        c_ = c1 // 2
        self.cv1 = Conv(c1, c_, 1, 1)
        self.cv2 = Conv(4 * c_, c2, 1, 1)
        self.m = nn.MaxPool2d(k, stride=1, padding=k // 2)

    def forward(self, x):
        x = self.cv1(x)
        m1 = self.m(x)
        m2 = self.m(m1)
        m3 = self.m(m2)
        return self.cv2(torch.cat((x, m1, m2, m3), 1))


class Attention(nn.Module):
    def __init__(self, dim, num_heads=8, attn_ratio=0.5):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.key_dim = int(self.head_dim * attn_ratio)
        self.scale = self.key_dim ** -0.5
        nh_kd = self.key_dim * num_heads
        h = dim + nh_kd * 2
        self.qkv = Conv(dim, h, 1, act=False)
        self.proj = Conv(dim, dim, 1, act=False)
        self.pe = Conv(dim, dim, 3, 1, g=dim, act=False)

    def forward(self, x):
        B, C, H, W = x.shape
        N = H * W
        qkv = self.qkv(x)
        q, k, v = qkv.view(B, self.num_heads, self.key_dim * 2 + self.head_dim, N).split(
            [self.key_dim, self.key_dim, self.head_dim], dim=2
        )
        attn = (q.transpose(-2, -1) @ k) * self.scale
        attn = attn.softmax(dim=-1)
        out = (v @ attn.transpose(-2, -1)).view(B, C, H, W) + self.pe(x)
        return self.proj(out)


class PSABlock(nn.Module):
    def __init__(self, c, attn_ratio=0.5, num_heads=4, shortcut=True):
        super().__init__()
        self.attn = Attention(c, attn_ratio=attn_ratio, num_heads=num_heads)
        self.ffn = nn.Sequential(Conv(c, c * 2, 1), Conv(c * 2, c, 1, act=False))
        self.add = shortcut

    def forward(self, x):
        x = x + self.attn(x) if self.add else self.attn(x)
        return x + self.ffn(x) if self.add else self.ffn(x)


class C2fPSA(nn.Module):
    def __init__(self, c1, c2, n=1, e=0.5):
        super().__init__()
        assert c1 == c2
        self.c = int(c1 * e)
        self.cv1 = Conv(c1, 2 * self.c, 1, 1)
        self.cv2 = Conv(2 * self.c, c1, 1)
        self.m = nn.Sequential(*(PSABlock(self.c, attn_ratio=0.5, num_heads=self.c // 64) for _ in range(n)))

    def forward(self, x):
        a, b = self.cv1(x).split((self.c, self.c), dim=1)
        b = self.m(b)
        return self.cv2(torch.cat((a, b), 1))


class SCDown(nn.Module):
    def __init__(self, c1, c2, k, s):
        super().__init__()
        self.cv1 = Conv(c1, c2, 1, 1)
        self.cv2 = Conv(c2, c2, k=k, s=s, g=c2, act=False)

    def forward(self, x):
        return self.cv2(self.cv1(x))


class Backbone(nn.Module):
    """Backbone trích xuất đặc trưng đa mức (p3, p4, p5)."""
    def __init__(self, w=(16, 32, 64, 128, 256), n=(1, 2, 2, 1)):
        super().__init__()
        c0, c1, c2, c3, c4 = w
        n0, n1, n2, n3 = n

        self.stem = Conv(3, c0, 3, 2)
        self.stage1 = nn.Sequential(
            Conv(c0, c1, 3, 2),
            C2f(c1, c1, n=n0, shortcut=True),
        )
        self.stage2 = nn.Sequential(
            Conv(c1, c2, 3, 2),
            C2f(c2, c2, n=n1, shortcut=True),
        )
        self.stage3 = nn.Sequential(
            SCDown(c2, c3, 3, 2),
            C2fCIB(c3, c3, n=n2, shortcut=True),
        )
        self.stage4 = nn.Sequential(
            SCDown(c3, c4, 3, 2),
            C2fCIB(c4, c4, n=n3, shortcut=True, large_kernel=True),
            SPPF(c4, c4, k=5),
            C2fPSA(c4, c4, n=2, e=0.5),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        p3 = self.stage2(x)
        p4 = self.stage3(p3)
        p5 = self.stage4(p4)
        return p3, p4, p5


class PAFPN(nn.Module):
    """Cổ tổng hợp đặc trưng đa tỷ lệ PAFPN."""
    def __init__(self, chs=(64, 128, 256), n=1):
        super().__init__()
        c3, c4, c5 = chs
        self.up = nn.Upsample(scale_factor=2, mode="nearest")

        # Top-down
        self.c2f_p4 = C2f(c5 + c4, c4, n=n, shortcut=True)
        self.c2f_p3 = C2f(c4 + c3, c3, n=n, shortcut=False)

        # Bottom-up
        self.down3 = Conv(c3, c3, 3, 2)
        self.c2f_n4 = C2fCIB(c3 + c4, c4, n=n, shortcut=True)

        self.down4 = SCDown(c4, c4, 3, 2)
        self.c2f_n5 = C2fCIB(c4 + c5, c5, n=n, shortcut=True, large_kernel=True)

    def forward(self, p3, p4, p5):
        # Top-down
        x = self.up(p5)
        x = torch.cat([x, p4], 1)
        p4_td = self.c2f_p4(x)

        x = self.up(p4_td)
        x = torch.cat([x, p3], 1)
        p3_out = self.c2f_p3(x)

        # Bottom-up
        x = self.down3(p3_out)
        x = torch.cat([x, p4_td], 1)
        p4_out = self.c2f_n4(x)

        x = self.down4(p4_out)
        x = torch.cat([x, p5], 1)
        p5_out = self.c2f_n5(x)

        return p3_out, p4_out, p5_out

print("[✓] Đã định nghĩa các khối CNN và kiến trúc Backbone PAFPN thành công!")

## 🤖 Section 4: Kiến trúc Không gian - Thời gian `ConvGRUClassifier` & `DrowsyClassifier`
Định nghĩa kiến trúc mô hình chính bảo toàn đặc trưng không gian 2D qua trục thời gian với cơ chế Chú ý Kép Dual Attention và mô hình End-to-End.

In [ ]:
# 1. Cổ giảm kênh không gian (SpatialReductionNeck)
class SpatialReductionNeck(nn.Module):
    def __init__(
            self,
            in_channels: Union[Tuple[Tuple[int, ...], ...], Tuple[int, ...]] = (64, 128, 256),
            out_channels: int = 128,
            dropout: float = 0.1,
            target_size: Tuple[int, int] = (14, 14)
    ):
        super().__init__()
        if len(in_channels) > 0 and isinstance(in_channels[0], (tuple, list)):
            self.channel_dims = tuple(c[0] for c in in_channels)
        else:
            self.channel_dims = tuple(int(c) for c in in_channels)

        self.num_scales = len(self.channel_dims)
        self.total_in_channels = sum(self.channel_dims)
        self.out_channels = int(out_channels)
        self.target_size = tuple(target_size)

        self.p3_down = nn.MaxPool2d(kernel_size=2, stride=2)
        self.p5_up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)

        self.reduction = nn.Sequential(
            nn.Conv2d(self.total_in_channels, self.out_channels, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(self.out_channels),
            nn.SiLU(inplace=True),
            nn.Dropout2d(p=dropout) if dropout > 0.0 else nn.Identity()
        )

    def _align_features(self, p3: torch.Tensor, p4: torch.Tensor, p5: torch.Tensor):
        th, tw = self.target_size
        out_p3 = self.p3_down(p3) if p3.shape[-2:] != (th, tw) else p3
        out_p4 = p4 if p4.shape[-2:] == (th, tw) else F.interpolate(p4, size=(th, tw), mode="bilinear", align_corners=False)
        out_p5 = self.p5_up(p5) if p5.shape[-2:] != (th, tw) else p5
        return out_p3, out_p4, out_p5

    def forward(self, features, seq_lens=None, *args):
        if len(args) > 0:
            features = (features, *args)
        if isinstance(features, (tuple, list)):
            p3, p4, p5 = features[0], features[1], features[2]
            is_5d = (p3.dim() == 5)
            orig_b, orig_t = 0, 0
            if is_5d:
                orig_b, orig_t = p3.shape[0], p3.shape[1]
                p3 = p3.reshape(orig_b * orig_t, *p3.shape[2:])
                p4 = p4.reshape(orig_b * orig_t, *p4.shape[2:])
                p5 = p5.reshape(orig_b * orig_t, *p5.shape[2:])

            out_p3, out_p4, out_p5 = self._align_features(p3, p4, p5)
            fused = torch.cat([out_p3, out_p4, out_p5], dim=1)

            if is_5d and seq_lens is not None:
                time_indices = torch.arange(orig_t, device=fused.device).unsqueeze(0)
                lens_expanded = seq_lens.to(fused.device).unsqueeze(1)
                mask = (time_indices < lens_expanded).view(-1)
                valid_fused = fused[mask]
                if valid_fused.numel() > 0:
                    valid_reduced = self.reduction(valid_fused)
                else:
                    valid_reduced = torch.empty((0, self.out_channels, self.target_size[0], self.target_size[1]),
                                                device=fused.device, dtype=fused.dtype)
                reduced = torch.zeros((orig_b * orig_t, self.out_channels, self.target_size[0], self.target_size[1]),
                                      device=fused.device, dtype=valid_reduced.dtype)
                reduced[mask] = valid_reduced
            else:
                reduced = self.reduction(fused)

            if is_5d:
                return reduced.reshape(orig_b, orig_t, self.out_channels, self.target_size[0], self.target_size[1])
            return reduced

        elif isinstance(features, torch.Tensor):
            if features.dim() == 5:
                b, t, c, h, w = features.shape
                f_flat = features.reshape(b * t, c, h, w)
                if (h, w) != self.target_size:
                    f_flat = F.interpolate(f_flat, size=self.target_size, mode="bilinear", align_corners=False)
                reduced = self.reduction(f_flat)
                return reduced.reshape(b, t, self.out_channels, self.target_size[0], self.target_size[1])
            return self.reduction(features)
        raise TypeError("[SpatialReductionNeck] Đầu vào phải là tuple/list (p3, p4, p5) hoặc torch.Tensor.")


# 2. Tế bào ConvGRUCell và ConvGRU
class ConvGRUCell(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, kernel_size: Union[int, Tuple[int, int]] = 3, bias: bool = True):
        super().__init__()
        self.input_dim = int(input_dim)
        self.hidden_dim = int(hidden_dim)
        self.bias = bias
        self.kernel_size = (kernel_size, kernel_size) if isinstance(kernel_size, int) else tuple(kernel_size)
        self.padding = (self.kernel_size[0] // 2, self.kernel_size[1] // 2)

        self.conv_gates = nn.Conv2d(self.input_dim + self.hidden_dim, 2 * self.hidden_dim, self.kernel_size, padding=self.padding, bias=self.bias)
        self.conv_candidate = nn.Conv2d(self.input_dim + self.hidden_dim, self.hidden_dim, self.kernel_size, padding=self.padding, bias=self.bias)
        self.reset_parameters()

    def reset_parameters(self):
        with torch.no_grad():
            nn.init.kaiming_normal_(self.conv_gates.weight[:, :self.input_dim], mode="fan_out", nonlinearity="relu")
            nn.init.orthogonal_(self.conv_gates.weight[:, self.input_dim:])
            if self.conv_gates.bias is not None:
                nn.init.zeros_(self.conv_gates.bias)
                self.conv_gates.bias[self.hidden_dim:].fill_(1.0)
            nn.init.kaiming_normal_(self.conv_candidate.weight[:, :self.input_dim], mode="fan_out", nonlinearity="tanh")
            nn.init.orthogonal_(self.conv_candidate.weight[:, self.input_dim:])
            if self.conv_candidate.bias is not None:
                nn.init.zeros_(self.conv_candidate.bias)

    def init_hidden(self, batch_size: int, spatial_size: Tuple[int, int], device=None, dtype=None):
        target_device = device if device is not None else self.conv_gates.weight.device
        target_dtype = dtype if dtype is not None else self.conv_gates.weight.dtype
        return torch.zeros(batch_size, self.hidden_dim, spatial_size[0], spatial_size[1], device=target_device, dtype=target_dtype)

    def forward(self, input_tensor: torch.Tensor, h_cur: Optional[torch.Tensor] = None):
        batch_size, _, height, width = input_tensor.shape
        if h_cur is None:
            h_cur = self.init_hidden(batch_size, (height, width), device=input_tensor.device, dtype=input_tensor.dtype)
        combined = torch.cat([input_tensor, h_cur], dim=1)
        gates = self.conv_gates(combined)
        reset_raw, update_raw = torch.split(gates, self.hidden_dim, dim=1)
        reset_gate = torch.sigmoid(reset_raw)
        update_gate = torch.sigmoid(update_raw)

        h_cur_casted = h_cur.to(dtype=reset_gate.dtype)
        in_casted = input_tensor.to(dtype=reset_gate.dtype)
        combined_candidate = torch.cat([in_casted, reset_gate * h_cur_casted], dim=1)
        candidate = torch.tanh(self.conv_candidate(combined_candidate))
        h_next = (1.0 - update_gate) * h_cur_casted + update_gate * candidate
        return h_next


class ConvGRU(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: Union[int, List[int], Tuple[int, ...]], kernel_size=3, num_layers=2, batch_first=True, dropout=0.0):
        super().__init__()
        self.input_dim = int(input_dim)
        self.num_layers = int(num_layers)
        self.batch_first = batch_first
        self.hidden_dims = [int(hidden_dim)] * self.num_layers if isinstance(hidden_dim, int) else [int(h) for h in hidden_dim]

        cell_list, dropout_list = [], []
        for i in range(self.num_layers):
            cur_in = self.input_dim if i == 0 else self.hidden_dims[i - 1]
            cell_list.append(ConvGRUCell(cur_in, self.hidden_dims[i], kernel_size=kernel_size))
            dropout_list.append(nn.Dropout2d(p=dropout) if (dropout > 0 and i < self.num_layers - 1) else nn.Identity())

        self.cells = nn.ModuleList(cell_list)
        self.dropouts = nn.ModuleList(dropout_list)

    def forward(self, input_tensor: torch.Tensor, hidden_state=None, seq_lens=None):
        if not self.batch_first:
            input_tensor = input_tensor.permute(1, 0, 2, 3, 4)
        b, seq_len, _, h, w = input_tensor.shape
        if hidden_state is None:
            hidden_state = [cell.init_hidden(b, (h, w), device=input_tensor.device, dtype=input_tensor.dtype) for cell in self.cells]

        cur_layer_input = input_tensor
        last_states = []

        for layer_idx, cell in enumerate(self.cells):
            h_cur = hidden_state[layer_idx]
            output_inner = []
            for t in range(seq_len):
                x_t = cur_layer_input[:, t, :, :, :]
                h_next = cell(input_tensor=x_t, h_cur=h_cur)
                if seq_lens is not None:
                    mask = (seq_lens > t).view(b, 1, 1, 1).to(dtype=h_next.dtype)
                    h_cur = mask * h_next + (1.0 - mask) * h_cur
                else:
                    h_cur = h_next
                output_inner.append(h_cur.unsqueeze(1))

            layer_output = torch.cat(output_inner, dim=1)
            if not isinstance(self.dropouts[layer_idx], nn.Identity):
                b_s, t_s, c_s, h_s, w_s = layer_output.shape
                layer_output = self.dropouts[layer_idx](layer_output.view(b_s * t_s, c_s, h_s, w_s)).view(b_s, t_s, c_s, h_s, w_s)
            cur_layer_input = layer_output
            last_states.append(h_cur)

        return cur_layer_input, last_states


# 3. Dual Attention Modules
class SpatialAttentionPooling(nn.Module):
    def __init__(self, in_channels: int):
        super().__init__()
        self.conv_attn = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // 2, kernel_size=1, bias=False),
            nn.BatchNorm2d(in_channels // 2),
            nn.SiLU(inplace=True),
            nn.Conv2d(in_channels // 2, 1, kernel_size=1, bias=True)
        )

    def forward(self, x: torch.Tensor, seq_lens=None, return_weights=False):
        b, t, c, h, w = x.shape
        x_flat = x.view(b * t, c, h, w)
        scores = self.conv_attn(x_flat).view(b * t, 1, h * w)
        weights = F.softmax(scores, dim=-1)
        x_reshaped = x_flat.view(b * t, c, h * w)
        pooled = torch.bmm(x_reshaped, weights.transpose(1, 2)).view(b, t, c)
        if seq_lens is not None:
            time_idx = torch.arange(t, device=x.device).unsqueeze(0)
            mask = (time_idx < seq_lens.to(x.device).unsqueeze(1)).unsqueeze(-1)
            pooled = pooled * mask.to(pooled.dtype)
        if return_weights:
            return pooled, weights.view(b, t, 1, h, w)
        return pooled


class TemporalAttentionPooling(nn.Module):
    def __init__(self, in_features: int, hidden_dim: int = 64):
        super().__init__()
        self.attn_net = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1, bias=False)
        )

    def forward(self, x: torch.Tensor, seq_lens=None):
        b, t, c = x.shape
        scores = self.attn_net(x).squeeze(-1)  # [B, T]
        if seq_lens is not None:
            time_idx = torch.arange(t, device=x.device).unsqueeze(0)
            mask = time_idx >= seq_lens.to(x.device).unsqueeze(1)
            scores = scores.masked_fill(mask, -1e4)
        weights = F.softmax(scores, dim=1).unsqueeze(1)  # [B, 1, T]
        pooled = torch.bmm(weights, x).squeeze(1)       # [B, C]
        return pooled, weights.squeeze(1)


# 4. SpatioTemporal ConvGRUClassifier
class ConvGRUClassifier(nn.Module):
    def __init__(
            self,
            input_dim: int = 128,
            hidden_dim: Union[int, List[int], Tuple[int, ...]] = 128,
            num_layers: int = 2,
            kernel_size: int = 3,
            num_classes: int = 2,
            spatial_in_channels: Tuple[int, int, int] = (64, 128, 256),
            target_size: Tuple[int, int] = (14, 14),
            neck_dropout: float = 0.25,
            convgru_dropout: float = 0.1,
            dropout: float = 0.35,
            supervision_mode: str = "attention_pooling"
    ):
        super().__init__()
        self.num_classes = int(num_classes)
        self.supervision_mode = str(supervision_mode).lower()

        self.spatial_neck = SpatialReductionNeck(
            in_channels=spatial_in_channels,
            out_channels=input_dim,
            dropout=neck_dropout,
            target_size=target_size
        )
        self.convgru = ConvGRU(
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            kernel_size=kernel_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=convgru_dropout
        )
        last_hid_dim = hidden_dim[-1] if isinstance(hidden_dim, (list, tuple)) else hidden_dim
        self.spatial_pooling = SpatialAttentionPooling(in_channels=last_hid_dim)
        self.temporal_pooling = TemporalAttentionPooling(in_features=last_hid_dim)
        self.fc_out = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(last_hid_dim, num_classes)
        )

    def forward(self, features, seq_lens=None, hidden_state=None, return_sequence=False, return_weights=False, return_state=False, *args):
        if len(args) > 0 and isinstance(features, torch.Tensor):
            features = (features, *args)
        fused = self.spatial_neck(features, seq_lens=seq_lens)
        is_single_frame = (fused.dim() == 4)
        if is_single_frame:
            fused = fused.unsqueeze(1)

        st_out, last_states = self.convgru(fused, hidden_state=hidden_state, seq_lens=seq_lens)
        if return_weights:
            t_features, s_weights = self.spatial_pooling(st_out, seq_lens=seq_lens, return_weights=True)
        else:
            t_features = self.spatial_pooling(st_out, seq_lens=seq_lens, return_weights=False)
            s_weights = None

        if return_sequence or (self.supervision_mode in ("sequence", "frame")):
            logits = self.fc_out(t_features)
            t_weights = None
            if is_single_frame:
                logits = logits.squeeze(1)
        else:
            pooled, t_weights = self.temporal_pooling(t_features, seq_lens=seq_lens)
            logits = self.fc_out(pooled)

        outputs = [logits]
        if return_weights:
            outputs.append({"spatial": s_weights, "temporal": t_weights})
        if return_state:
            outputs.append(last_states)
        return tuple(outputs) if len(outputs) > 1 else logits

print("[✓] Đã định nghĩa lớp ConvGRUClassifier thành công!")

In [ ]:
# 5. Mô hình Hợp nhất End-to-End DrowsyClassifier (src/models1.py)
class DrowsyClassifier(nn.Module):
    """
    Mô hình Phân loại Trạng thái Buồn ngủ Toàn diện End-to-End Spatio-Temporal.

    Luồng xử lý (Data Flow):
      Video [B, T, 3, H, W]
        -> Reshape [B*T, 3, H, W]
        -> Mini-Chunking Backbone & PAFPN (Gradient Checkpointing per-chunk & Padding-Aware Masking)
        -> Trích xuất 3 mức đặc trưng không gian: p3, p4, p5
        -> Reshape [B, T, C, H_feat, W_feat]
        -> ConvGRUClassifier (SpatialReductionNeck -> ConvGRU -> Dual Attention -> FC Head)
        -> Logits phân loại buồn ngủ.
    """

    def __init__(
            self,
            num_classes: int = 2,
            chunk_size: int = 32,
            backbone_w: Tuple[int, ...] = (16, 32, 64, 128, 256),
            backbone_n: Tuple[int, ...] = (1, 2, 2, 1),
            neck_n: int = 1,
            input_dim: int = 128,
            hidden_dim: Union[int, Tuple[int, ...], List[int]] = (128, 128),
            target_size: Tuple[int, int] = (14, 14),
            supervision_mode: str = "attention_pooling",
            gradient_checkpointing: bool = False,
            freeze_bn: bool = True,
    ):
        """
        Khởi tạo DrowsyClassifier End-to-End.

        Args:
            num_classes (int): Số lớp phân loại (mặc định 2: Tỉnh táo / Buồn ngủ).
            chunk_size (int): Kích thước mini-chunk khi trích xuất qua Backbone/PAFPN (mặc định 32 frames).
                              Giúp khống chế đỉnh VRAM ở O(chunk_size) thay vì O(B*T).
            backbone_w (Tuple[int, ...]): Độ rộng các kênh của Backbone (mặc định (16, 32, 64, 128, 256)).
            backbone_n (Tuple[int, ...]): Số lượng khối lặp lại trong từng tầng Backbone.
            neck_n (int): Số lượng khối lặp lại trong PAFPN.
            input_dim (int): Số kênh nén không gian đầu vào của ConvGRU (mặc định 128).
            hidden_dim (Union[int, Tuple, List]): Số kênh ẩn trong các tầng ConvGRU.
            target_size (Tuple[int, int]): Kích thước lưới không gian hội tụ (mặc định (14, 14)).
            supervision_mode (str): 'attention_pooling' (clip-level) hoặc 'sequence' (frame-level).
            gradient_checkpointing (bool): Bật checkpointing cho Backbone/PAFPN để tiết kiệm VRAM khi train.
            freeze_bn (bool): Khóa thống kê running mean/var của BatchNorm trong Backbone/PAFPN (mặc định True).
        """
        super().__init__()
        self.num_classes = int(num_classes)
        self.chunk_size = max(0, int(chunk_size)) if chunk_size is not None else 0
        self.backbone_w = tuple(backbone_w)
        self.backbone_n = tuple(backbone_n)
        self.neck_n = int(neck_n)
        self.input_dim = int(input_dim)
        self.hidden_dim = hidden_dim
        self.target_size = tuple(target_size)
        self.supervision_mode = str(supervision_mode).lower()
        self.gradient_checkpointing = bool(gradient_checkpointing)
        self.freeze_bn = bool(freeze_bn)

        # 1. Trích xuất đặc trưng không gian (Spatial Backbone)
        self.backbone = Backbone(w=self.backbone_w, n=self.backbone_n)
        c3, c4, c5 = self.backbone_w[2], self.backbone_w[3], self.backbone_w[4]
        self.neck_chs = (c3, c4, c5)

        # 2. Cổ tổng hợp đặc trưng đa tỷ lệ (Feature Aggregation Neck)
        self.neck = PAFPN(chs=self.neck_chs, n=self.neck_n)

        # 3. Bộ phân loại không gian - thời gian (Spatio-Temporal Classifier)
        self.conv_gru = ConvGRUClassifier(
            input_dim=self.input_dim,
            hidden_dim=self.hidden_dim,
            spatial_in_channels=self.neck_chs,
            target_size=self.target_size,
            num_classes=self.num_classes,
            supervision_mode=self.supervision_mode,
        )

        # 4. Tự động áp dụng khóa BatchNorm ngay khi khởi tạo nếu freeze_bn=True
        if self.freeze_bn:
            self._freeze_backbone_neck_bn()

    def train(self, mode: bool = True):
        """Chuyển đổi trạng thái train/eval kèm hỗ trợ khóa BatchNorm tự động nếu freeze_bn=True."""
        super().train(mode)
        if mode and self.freeze_bn:
            self._freeze_backbone_neck_bn()
        return self

    def _freeze_backbone_neck_bn(self) -> None:
        """Đóng băng thống kê Batch Normalization trong Backbone và Neck để tránh ô nhiễm bởi video frames."""
        from torch.nn.modules.batchnorm import _BatchNorm
        for m in self.backbone.modules():
            if isinstance(m, _BatchNorm):
                m.eval()
        for m in self.neck.modules():
            if isinstance(m, _BatchNorm):
                m.eval()

    def _unfreeze_backbone_neck_bn(self) -> None:
        """Mở khóa thống kê Batch Normalization trong Backbone và Neck khi freeze_bn=False."""
        from torch.nn.modules.batchnorm import _BatchNorm
        for m in self.backbone.modules():
            if isinstance(m, _BatchNorm):
                m.train()
        for m in self.neck.modules():
            if isinstance(m, _BatchNorm):
                m.train()

    def _forward_backbone_neck_chunked(
            self,
            flat_frames: torch.Tensor,
            valid_mask: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Lan truyền qua Backbone và PAFPN theo cơ chế Mini-Chunking.

        Đảm bảo 4 tiêu chí cốt lõi:
        1. Bảo toàn tích tụ Gradient (Gradient Accumulation): Kết quả ghép qua torch.cat,
           Autograd tự động cộng dồn chính xác dL/dW vào các trọng số dùng chung.
        2. Xử lý Padding an toàn (Padding-Aware Masking): Khung hình zero-padding được lọc
           bỏ trước khi qua Backbone, không sinh đạo hàm và phục hồi (scatter) zero chuẩn xác.
        3. Bảo vệ thống kê BatchNorm: Khung hình padding không bao giờ đi vào BatchNorm.
           Khi freeze_bn=True, BN ở eval mode hoạt động như phép biến đổi affine cố định.
        4. Tối ưu hóa Gradient Checkpointing: Checkpoint độc lập trên từng chunk, giải phóng
           activation ngay sau backward của chunk đó.
        """
        total_frames = flat_frames.shape[0]
        if total_frames == 0:
            c3, c4, c5 = self.neck_chs
            return (
                torch.empty((0, c3, 28, 28), device=flat_frames.device, dtype=flat_frames.dtype),
                torch.empty((0, c4, 14, 14), device=flat_frames.device, dtype=flat_frames.dtype),
                torch.empty((0, c5, 7, 7), device=flat_frames.device, dtype=flat_frames.dtype),
            )

        # 1. Lọc khung hình hợp lệ nếu có mặt nạ (loại bỏ padding khung hình)
        if valid_mask is not None and not bool(valid_mask.all()):
            active_frames = flat_frames[valid_mask]
            needs_scatter = True
        else:
            active_frames = flat_frames
            needs_scatter = False

        num_active = active_frames.shape[0]
        if num_active == 0:
            c3, c4, c5 = self.neck_chs
            h3, w3 = self.target_size[0] * 2, self.target_size[1] * 2
            h4, w4 = self.target_size[0], self.target_size[1]
            h5, w5 = max(1, self.target_size[0] // 2), max(1, self.target_size[1] // 2)
            return (
                torch.zeros((total_frames, c3, h3, w3), device=flat_frames.device, dtype=flat_frames.dtype),
                torch.zeros((total_frames, c4, h4, w4), device=flat_frames.device, dtype=flat_frames.dtype),
                torch.zeros((total_frames, c5, h5, w5), device=flat_frames.device, dtype=flat_frames.dtype),
            )

        # 2. Định nghĩa hàm forward cho từng stage/chunk
        def _stage_fwd(f: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
            o3, o4, o5 = self.backbone(f)
            return self.neck(o3, o4, o5)

        # 3. Chia nhỏ thành các mini-chunk
        step = self.chunk_size if (self.chunk_size > 0 and self.chunk_size < num_active) else num_active
        p3_list, p4_list, p5_list = [], [], []

        for i in range(0, num_active, step):
            chunk = active_frames[i : min(i + step, num_active)]
            is_single = (chunk.shape[0] <= 1)

            # Phòng vệ: Nếu freeze_bn=False nhưng chunk <= 1 frame (do padding lẻ),
            # tạm thời chuyển BN sang eval cho chunk đó để bảo vệ phân phối không bị suy biến
            single_frame_guarded = False
            if is_single and not self.freeze_bn and self.training:
                self._freeze_backbone_neck_bn()
                single_frame_guarded = True

            if self.gradient_checkpointing and self.training:
                if not chunk.requires_grad:
                    chunk.requires_grad_(True)
                o3, o4, o5 = torch.utils.checkpoint.checkpoint(_stage_fwd, chunk, use_reentrant=False)
            else:
                o3, o4, o5 = _stage_fwd(chunk)

            if single_frame_guarded:
                self._unfreeze_backbone_neck_bn()

            p3_list.append(o3)
            p4_list.append(o4)
            p5_list.append(o5)

        p3_active = torch.cat(p3_list, dim=0)
        p4_active = torch.cat(p4_list, dim=0)
        p5_active = torch.cat(p5_list, dim=0)

        # 4. Phục hồi khung hình padding (Scatter) nếu có lọc
        if needs_scatter:
            c3, c4, c5 = self.neck_chs
            h3, w3 = p3_active.shape[-2:]
            h4, w4 = p4_active.shape[-2:]
            h5, w5 = p5_active.shape[-2:]

            p3 = torch.zeros((total_frames, c3, h3, w3), device=flat_frames.device, dtype=p3_active.dtype)
            p4 = torch.zeros((total_frames, c4, h4, w4), device=flat_frames.device, dtype=p4_active.dtype)
            p5 = torch.zeros((total_frames, c5, h5, w5), device=flat_frames.device, dtype=p5_active.dtype)

            p3[valid_mask] = p3_active
            p4[valid_mask] = p4_active
            p5[valid_mask] = p5_active
        else:
            p3, p4, p5 = p3_active, p4_active, p5_active

        return p3, p4, p5

    def _forward_backbone_neck(
            self,
            flat_frames: torch.Tensor,
            valid_mask: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Hàm bọc tương thích ngược, chuyển tiếp sang _forward_backbone_neck_chunked."""
        return self._forward_backbone_neck_chunked(flat_frames, valid_mask=valid_mask)

    def forward(
            self,
            x: torch.Tensor,
            seq_lens: Optional[torch.Tensor] = None,
            hidden_state: Optional[List[torch.Tensor]] = None,
            return_sequence: bool = False,
            return_weights: bool = False,
            return_state: bool = False,
    ) -> Union[torch.Tensor, Tuple[Any, ...]]:
        """
        Lan truyền xuôi toàn bộ mô hình End-to-End.

        Args:
            x (torch.Tensor): Tensor chuỗi video 5D dạng [Batch, Time, Channels, Height, Width].
                             Chấp nhận kiểu dữ liệu uint8 [0..255] hoặc float32 [0.0..1.0].
            seq_lens (Optional[torch.Tensor]): Độ dài thực tế của từng clip trong batch [Batch].
            hidden_state (Optional[List[torch.Tensor]]): Hidden state ngoại vi cho chế độ streaming.
            return_sequence (bool): Nếu True, trả về logits từng frame [B, T, num_classes].
            return_weights (bool): Nếu True, trả về dict trọng số chú ý không gian và thời gian.
            return_state (bool): Nếu True, trả về hidden state cuối cùng của ConvGRU.

        Returns:
            logits (torch.Tensor) hoặc Tuple chứa (logits, [weights], [states]).
        """
        if x.dim() != 5:
            raise ValueError(
                f"[DrowsyClassifier] Yêu cầu đầu vào Tensor 5D [B, T, C, H, W], nhưng nhận {x.dim()}D với shape {tuple(x.shape)}"
            )

        B, T, C, H, W = x.shape

        # 1. Tự động chuyển đổi uint8 sang float32 [0.0, 1.0] an toàn, không blocking GPU sync
        if x.dtype == torch.uint8:
            x = x.float().div_(255.0)

        # 2. Xây dựng mặt nạ hợp lệ nếu có seq_lens (loại bỏ padding khung hình video ngắn)
        valid_mask = None
        if seq_lens is not None:
            time_indices = torch.arange(T, device=x.device).unsqueeze(0)  # [1, T]
            lens_expanded = seq_lens.to(x.device).unsqueeze(1)            # [B, 1]
            valid_mask = (time_indices < lens_expanded).view(-1)          # [B * T]

        # 3. Định hình lại không gian an toàn cho cả non-contiguous tensors
        flat_frames = x.reshape(B * T, C, H, W)

        # 4. Trích xuất đặc trưng qua Backbone & PAFPN theo Mini-Chunking
        out3, out4, out5 = self._forward_backbone_neck_chunked(flat_frames, valid_mask=valid_mask)

        # 5. Định hình lại không gian - thời gian cho 3 tầng đặc trưng đa tỷ lệ
        p3 = out3.reshape(B, T, *out3.shape[1:])
        p4 = out4.reshape(B, T, *out4.shape[1:])
        p5 = out5.reshape(B, T, *out5.shape[1:])

        # 6. Phân loại qua SpatioTemporal ConvGRUClassifier
        outputs = self.conv_gru(
            (p3, p4, p5),
            seq_lens=seq_lens,
            hidden_state=hidden_state,
            return_sequence=return_sequence,
            return_weights=return_weights,
            return_state=return_state,
        )

        return outputs


print("[✓] Đã định nghĩa lớp DrowsyClassifier End-to-End thành công!")


## 📹 Section 5: Tăng cường Dữ liệu Video & Pipeline Nạp Video Thô
1. `VideoAugmenter`: Bộ tăng cường chuỗi khung hình từ `src/augment.py`. Sử dụng **CHUNG 1 RANDOM SEED** cho toàn bộ các frame trong 1 clip video để bảo toàn tính nhất quán thời gian (Temporal Consistency).
2. `RawVideoFramesDataset`: Giải mã video trực tiếp qua OpenCV, lấy mẫu đều đặn 0.2s, tạo tensor `uint8` siêu nhẹ trên CPU.
3. `collate_video_frames`: Ghép nối các chuỗi video có độ dài động và tạo mask đệm an toàn.

In [ ]:
# 1. Bộ tăng cường chuỗi video nhất quán thời gian (VideoAugmenter từ src/augment.py)
def _make_gauss_noise(var_limit, p):
    params = inspect.signature(A.GaussNoise.__init__).parameters
    if "var_limit" in params:
        return A.GaussNoise(var_limit=var_limit, p=p)
    if "std_range" in params:
        var_min, var_max = var_limit
        std_min = max(0.0, min(1.0, (var_min ** 0.5) / 255.0))
        std_max = max(0.0, min(1.0, (var_max ** 0.5) / 255.0))
        return A.GaussNoise(std_range=(std_min, std_max), p=p)
    return A.GaussNoise(p=p)


def _make_shift_scale_rotate(shift_limit, scale_limit, rotate_limit, p, fill_color=(114, 114, 114)):
    if hasattr(A, "Affine"):
        scale = (1.0 - scale_limit, 1.0 + scale_limit) if isinstance(scale_limit, (int, float)) else (
            1.0 + scale_limit[0], 1.0 + scale_limit[1])
        translate = (-shift_limit, shift_limit) if isinstance(shift_limit, (int, float)) else shift_limit
        rotate = (-rotate_limit, rotate_limit) if isinstance(rotate_limit, (int, float)) else rotate_limit
        return A.Affine(scale=scale, translate_percent=translate, rotate=rotate, border_mode=cv2.BORDER_CONSTANT,
                        fill=fill_color, p=p)
    params = inspect.signature(A.ShiftScaleRotate.__init__).parameters
    kwargs = dict(shift_limit=shift_limit, scale_limit=scale_limit, rotate_limit=rotate_limit,
                  border_mode=cv2.BORDER_CONSTANT, p=p)
    kwargs["value" if "value" in params else "fill"] = fill_color
    return A.ShiftScaleRotate(**kwargs)


class VideoAugmenter:
    """
    Pipeline tăng cường dữ liệu cho chuỗi khung hình video.
    Bảo toàn tính nhất quán thời gian bằng cách áp dụng chung 1 random seed cho tất cả các frames trong cùng một clip.
    """
    def __init__(self, cfg: Optional[dict] = None):
        default_cfg = {
            "horizontalFlip": 0.5,
            "shiftScaleRotate": (0.08, 0.10, 15, 0.8),
            "randomBrightnessContrast": 0.4,
            "hueSaturationValue": (15, 20, 15, 0.3),
            "gaussNoise": (5.0, 25.0, 0.25),
            "blur": (5, 0.15),
            "p": 0.5
        }
        self.cfg = cfg or default_cfg
        shift_limit, scale_limit, rotate_limit, ssr_p = self.cfg["shiftScaleRotate"]
        hue_shift, sat_shift, val_shift, hsv_p = self.cfg["hueSaturationValue"]
        var_min, var_max, gn_p = self.cfg["gaussNoise"]
        blur_limit, blur_p = self.cfg["blur"]

        self.transform = A.Compose(
            [
                A.HorizontalFlip(p=self.cfg["horizontalFlip"]),
                _make_shift_scale_rotate(shift_limit, scale_limit, rotate_limit, ssr_p),
                A.RandomBrightnessContrast(p=self.cfg["randomBrightnessContrast"]),
                A.HueSaturationValue(hue_shift_limit=hue_shift, sat_shift_limit=sat_shift, val_shift_limit=val_shift,
                                     p=hsv_p),
                _make_gauss_noise((var_min, var_max), p=gn_p),
                A.Blur(blur_limit=blur_limit, p=blur_p),
            ],
            p=self.cfg["p"]
        )

    def set_seed(self, seed: int):
        random.seed(seed)
        np.random.seed(seed)
        if hasattr(self.transform, "set_random_seed"):
            self.transform.set_random_seed(seed)

    def apply_sequence(self, frames: List[np.ndarray], seed: Optional[int] = None) -> List[np.ndarray]:
        if seed is None:
            seed = random.randint(0, 2 ** 31 - 1)
        aug_frames = []
        for frame in frames:
            self.set_seed(seed)
            try:
                res = self.transform(image=frame)
                aug_frames.append(res["image"])
            except Exception:
                aug_frames.append(frame)
        return aug_frames

print("[✓] Đã định nghĩa lớp VideoAugmenter từ src/augment.py thành công!")

In [ ]:
# 2. Pipeline tiền xử lý và nạp dữ liệu Video Thô
def letterbox(image: np.ndarray, new_size: int = 224, color: Tuple[int, int, int] = (114, 114, 114)) -> np.ndarray:
    """Resize ảnh giữ nguyên tỉ lệ khung hình (aspect ratio) với padding đồng màu."""
    h, w = image.shape[:2]
    scale = min(new_size / h, new_size / w)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    canvas = np.full((new_size, new_size, 3), color, dtype=image.dtype)
    pad_left = (new_size - new_w) // 2
    pad_top = (new_size - new_h) // 2
    canvas[pad_top: pad_top + new_h, pad_left: pad_left + new_w] = resized
    return canvas


@dataclass
class RawVideoSample:
    path: Path
    video_id: str
    split: str
    label: int
    label_name: str
    source_dataset: str = "custom"
    subject_id: Optional[str] = None


class RawVideoFramesDataset(Dataset):
    """
    Dataset đọc trực tiếp video thô (.mp4, .avi, .mkv) bằng OpenCV.
    Chạy 100% trên CPU, trả về Tensor uint8 tiết kiệm RAM và bảo đảm an toàn đa tiến trình.
    """
    def __init__(
            self,
            dataset_dir: Union[str, Path],
            manifest_file: Optional[str] = "dataset_merged_split.csv",
            split: str = "train",
            sample_interval: float = 0.2,
            seq_len: Optional[int] = None,
            video_exts: Tuple[str, ...] = (".avi", ".mp4", ".mkv", ".mov"),
            min_frames: int = 10,
            use_augmentation: bool = True,
            augmenter_cfg: Optional[dict] = None,
            img_size: int = 224
    ):
        super().__init__()
        self.dataset_dir = Path(dataset_dir)
        self.split = split.lower().strip()
        self.sample_interval = float(sample_interval)
        self.seq_len = seq_len
        self.video_exts = tuple(ext.lower() for ext in video_exts)
        self.min_frames = int(min_frames)
        self.use_augmentation = bool(use_augmentation and (self.split == "train"))
        self.img_size = int(img_size)

        self.augmenter = VideoAugmenter(augmenter_cfg) if self.use_augmentation else None
        self.samples: List[RawVideoSample] = []
        self._load_samples(manifest_file)

    def _load_samples(self, manifest_file: Optional[str]):
        manifest_path = None
        if manifest_file:
            cand = Path(manifest_file)
            if cand.exists():
                manifest_path = cand
            elif (self.dataset_dir / manifest_file).exists():
                manifest_path = self.dataset_dir / manifest_file

        if manifest_path and manifest_path.exists():
            with open(manifest_path, mode="r", encoding="utf-8") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    r_split = row.get("split", "").lower().strip()
                    if r_split != self.split:
                        continue
                    v_path_str = row.get("processed_path", "") or row.get("filepath", "") or row.get("path", "")
                    v_path = Path(v_path_str)
                    if not v_path.is_absolute():
                        if (self.dataset_dir / v_path).exists():
                            v_path = self.dataset_dir / v_path
                        elif (self.dataset_dir / self.split / v_path.name).exists():
                            v_path = self.dataset_dir / self.split / v_path.name

                    if not v_path.exists():
                        continue

                    label_val = int(row.get("binary_label", row.get("target", 0)))
                    label_name = row.get("label_name", "Drowsy" if label_val == 1 else "Alert")
                    self.samples.append(RawVideoSample(
                        path=v_path,
                        video_id=v_path.stem,
                        split=self.split,
                        label=label_val,
                        label_name=label_name
                    ))
        else:
            split_dir = self.dataset_dir / self.split
            search_dir = split_dir if split_dir.exists() else self.dataset_dir
            for cls_idx, cls_name in enumerate(["Alert", "Drowsy"]):
                for folder_name in [cls_name, cls_name.lower(), str(cls_idx)]:
                    c_dir = search_dir / folder_name
                    if c_dir.exists():
                        for f in c_dir.iterdir():
                            if f.suffix.lower() in self.video_exts:
                                self.samples.append(RawVideoSample(
                                    path=f,
                                    video_id=f.stem,
                                    split=self.split,
                                    label=cls_idx,
                                    label_name=cls_name
                                ))

    def __len__(self) -> int:
        return len(self.samples)

    def _extract_frames_opencv(self, video_path: Path) -> List[np.ndarray]:
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            return []
        fps = cap.get(cv2.CAP_PROP_FPS)
        fps = fps if (fps and fps > 0 and not math.isnan(fps)) else 25.0
        step_frames = max(1, int(round(fps * self.sample_interval)))

        frames = []
        frame_idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if frame_idx % step_frames == 0:
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame_resized = letterbox(frame_rgb, new_size=self.img_size)
                frames.append(frame_resized)
            frame_idx += 1
        cap.release()
        return frames

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int, int, Dict[str, Any]]:
        sample = self.samples[idx]
        frames = self._extract_frames_opencv(sample.path)

        if len(frames) < self.min_frames:
            if len(frames) == 0:
                dummy = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)
                frames = [dummy] * self.min_frames
            else:
                while len(frames) < self.min_frames:
                    frames.append(frames[-1].copy())

        if self.seq_len is not None:
            if len(frames) > self.seq_len:
                step = len(frames) / float(self.seq_len)
                indices = [int(round(i * step)) for i in range(self.seq_len)]
                frames = [frames[min(i, len(frames) - 1)] for i in indices]
            elif len(frames) < self.seq_len:
                while len(frames) < self.seq_len:
                    frames.append(frames[-1].copy())

        if self.use_augmentation and self.augmenter is not None:
            frames = self.augmenter.apply_sequence(frames)

        # Tensor uint8 CPU [T, 3, H, W]
        arr = np.stack(frames, axis=0)  # [T, H, W, 3]
        tensor = torch.from_numpy(arr).permute(0, 3, 1, 2).contiguous()  # [T, 3, H, W]
        seq_len = tensor.size(0)

        meta = {
            "video_id": sample.video_id,
            "path": str(sample.path),
            "label_name": sample.label_name
        }
        return tensor, sample.label, seq_len, meta


def collate_video_frames(batch: List[Tuple[torch.Tensor, int, int, Dict[str, Any]]]):
    valid_batch = [item for item in batch if item[0] is not None and item[0].size(0) > 0]
    if not valid_batch:
        return torch.empty(0), torch.empty(0, dtype=torch.long), torch.empty(0, dtype=torch.long), []

    tensors, labels, lengths, metas = zip(*valid_batch)
    batch_size = len(tensors)
    max_len = max(t.size(0) for t in tensors)
    _, c, h, w = tensors[0].shape

    padded = torch.zeros((batch_size, max_len, c, h, w), dtype=torch.uint8)
    for i, t in enumerate(tensors):
        padded[i, :t.size(0)] = t

    labels_tensor = torch.tensor(labels, dtype=torch.long)
    lens_tensor = torch.tensor(lengths, dtype=torch.long)
    return padded, labels_tensor, lens_tensor, list(metas)


def build_raw_video_dataloaders(
        dataset_dir: Union[str, Path],
        manifest_file: Optional[str] = "dataset_merged_split.csv",
        sample_interval: float = 0.2,
        seq_len: Optional[int] = None,
        batch_size: int = 2,
        val_batch_size: int = 2,
        num_workers: int = 2,
        val_num_workers: int = 2,
        pin_memory: bool = False,
        shuffle_train: bool = True,
        use_augmentation: bool = True,
        min_frames: int = 10,
        img_size: int = 224
) -> Tuple[DataLoader, DataLoader]:
    train_ds = RawVideoFramesDataset(
        dataset_dir=dataset_dir,
        manifest_file=manifest_file,
        split="train",
        sample_interval=sample_interval,
        seq_len=seq_len,
        min_frames=min_frames,
        use_augmentation=use_augmentation,
        img_size=img_size
    )
    val_ds = RawVideoFramesDataset(
        dataset_dir=dataset_dir,
        manifest_file=manifest_file,
        split="val",
        sample_interval=sample_interval,
        seq_len=seq_len,
        min_frames=min_frames,
        use_augmentation=False,
        img_size=img_size
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=shuffle_train,
        num_workers=num_workers,
        pin_memory=pin_memory,
        collate_fn=collate_video_frames
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=val_batch_size,
        shuffle=False,
        num_workers=val_num_workers,
        pin_memory=pin_memory,
        collate_fn=collate_video_frames
    )
    return train_loader, val_loader

print("[✓] Đã định nghĩa DataLoader và RawVideoFramesDataset thành công!")

## 🛠️ Section 6: Hạ tầng Huấn luyện & Cơ chế Tiếp tục Huấn luyện (`Trainer`)
Định nghĩa đầy đủ bộ điều phối `Trainer` chuyên biệt cho `DrowsyClassifier` End-to-End từ `src/train1.py`:
- Tính toán Accuracy, Recall (macro), F1-Score (macro).
- Cơ chế giám sát dừng sớm `EarlyStopping`.
- Khôi phục huấn luyện hoàn chỉnh `load_checkpoint` (Resumable Training & AMP).
- Cơ chế lưu tệp nguyên tử `_atomic_save` chống hỏng checkpoint.
- Tự động bắt lỗi ngoại lệ `CUDA Out-Of-Memory (OOM)` dọn cache và bỏ qua batch an toàn.

In [ ]:
# 1. Hàm tính toán Metrics và Early Stopping
def calculate_metrics(preds: np.ndarray, targets: np.ndarray) -> Dict[str, float]:
    """Tính toán Accuracy, Recall, F1-Score (macro) an toàn từ mảng NumPy."""
    if len(preds) == 0 or len(targets) == 0:
        return {"acc": 0.0, "recall": 0.0, "f1": 0.0}
    acc = accuracy_score(targets, preds)
    recall = recall_score(targets, preds, average='macro', zero_division=0)
    f1 = f1_score(targets, preds, average='macro', zero_division=0)
    return {"acc": float(acc), "recall": float(recall), "f1": float(f1)}


class EarlyStopping:
    """Bộ giám sát dừng sớm Early Stopping dựa trên metric chỉ định."""
    def __init__(self, patience: int = 10, min_delta: float = 1e-4, mode: str = "max", monitor: str = "val_f1"):
        self.patience = max(1, patience)
        self.min_delta = float(min_delta)
        self.mode = mode.lower().strip()
        self.monitor = monitor
        self.best_score = -float("inf") if self.mode == "max" else float("inf")
        self.counter = 0
        self.best_epoch = 0
        self.early_stop = False

    def step(self, current_val: float, epoch: int) -> bool:
        if self.mode == "max":
            improved = current_val > (self.best_score + self.min_delta)
        else:
            improved = current_val < (self.best_score - self.min_delta)

        if improved:
            self.best_score = current_val
            self.best_epoch = epoch
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop


# 2. Bộ điều phối Huấn luyện Trainer cho End-to-End DrowsyClassifier
class Trainer:
    """Pipeline Huấn luyện và Kiểm định Chuyên nghiệp cho End-to-End DrowsyClassifier."""
    def __init__(self, config: TrainConfig):
        self.cfg = config
        self.device = torch.device(self.cfg.device if torch.cuda.is_available() else "cpu")
        print(f"[*] Đang sử dụng thiết bị: {self.device}")

        # Kiểm tra tính tương thích của supervision_mode
        supervision = getattr(self.cfg, "supervision_mode", "attention_pooling")
        if supervision not in ("attention_pooling", "clip"):
            print(f"[CẢNH BÁO] supervision_mode='{supervision}' hiện chưa có nhãn frame-level. Đặt về 'attention_pooling'.")
            self.cfg.supervision_mode = "attention_pooling"

        # Khởi tạo DataLoader
        print("[*] Đang khởi tạo DataLoader từ dataset video thô...")
        val_batch_size = getattr(self.cfg, "val_batch_size", self.cfg.batch_size)
        val_num_workers = getattr(self.cfg, "val_num_workers", self.cfg.num_workers)
        min_frames = getattr(self.cfg, "min_frames", 10)

        img_size_cfg = getattr(self.cfg, "image_size", (224, 224))
        if isinstance(img_size_cfg, (tuple, list)):
            img_size = img_size_cfg[0]
            h_in, w_in = img_size_cfg[0], img_size_cfg[1]
        else:
            img_size = int(img_size_cfg)
            h_in, w_in = img_size, img_size

        self.train_loader, self.val_loader = build_raw_video_dataloaders(
            dataset_dir=self.cfg.dataset_dir,
            manifest_file=self.cfg.manifest_file,
            sample_interval=self.cfg.sample_interval,
            seq_len=self.cfg.seq_len,
            batch_size=self.cfg.batch_size,
            val_batch_size=val_batch_size,
            num_workers=self.cfg.num_workers,
            val_num_workers=val_num_workers,
            pin_memory=self.cfg.pin_memory,
            shuffle_train=self.cfg.shuffle,
            use_augmentation=self.cfg.use_augmentation,
            min_frames=min_frames,
            img_size=img_size
        )

        train_samples_count = len(self.train_loader.dataset)
        val_samples_count = len(self.val_loader.dataset)
        print(f"[*] Tập Train: {train_samples_count} video | Tập Val: {val_samples_count} video")

        if hasattr(self.cfg, "target_size") and self.cfg.target_size is not None:
            target_size = tuple(self.cfg.target_size)
        else:
            target_size = (max(1, h_in // 16), max(1, w_in // 16))

        # Khởi tạo mô hình DrowsyClassifier End-to-End từ đầu (Train from scratch)
        print(f"[*] Đang khởi tạo DrowsyClassifier (target_size={target_size}, img_size=({h_in}, {w_in}))...")
        num_classes = getattr(self.cfg, "num_classes", 2)
        backbone_w = getattr(self.cfg, "backbone_w", (16, 32, 64, 128, 256))
        backbone_n = getattr(self.cfg, "backbone_n", (1, 2, 2, 1))
        neck_n = getattr(self.cfg, "neck_n", 1)
        input_dim = getattr(self.cfg, "convgru_in_dim", getattr(self.cfg, "input_dim", 128))
        hidden_dim = getattr(self.cfg, "convgru_hidden_dim", getattr(self.cfg, "hidden_dim", 128))
        gradient_checkpointing = getattr(self.cfg, "gradient_checkpointing", True)
        freeze_bn = getattr(self.cfg, "freeze_bn", True)
        chunk_size = getattr(self.cfg, "chunk_size", 32)

        self.model = DrowsyClassifier(
            num_classes=num_classes,
            chunk_size=chunk_size,
            backbone_w=backbone_w,
            backbone_n=backbone_n,
            neck_n=neck_n,
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            target_size=target_size,
            supervision_mode=self.cfg.supervision_mode,
            gradient_checkpointing=gradient_checkpointing,
            freeze_bn=freeze_bn,
        ).to(self.device)

        total_params = sum(p.numel() for p in self.model.parameters())
        trainable_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        print(f"[*] Tổng số tham số DrowsyClassifier: {total_params:,} (Có thể huấn luyện: {trainable_params:,})")

        # Cấu hình Loss
        loss_type = self.cfg.loss_type.lower()
        if loss_type == "bce":
            if num_classes == 1:
                pos_weight = torch.tensor([self.cfg.pos_weight], device=self.device) if self.cfg.pos_weight else None
                self.criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction=self.cfg.bce_reduction)
                self.is_binary_bce = True
                print("[*] Sử dụng BCEWithLogitsLoss cho bài toán Binary Classification 1-Logit.")
            else:
                print(f"[CẢNH BÁO] Config yêu cầu loss_type='bce' nhưng num_classes={num_classes} (>1). Tự động dùng CrossEntropyLoss.")
                self.criterion = nn.CrossEntropyLoss()
                self.is_binary_bce = False
        else:
            self.criterion = nn.CrossEntropyLoss()
            self.is_binary_bce = False
            print("[*] Sử dụng CrossEntropyLoss chuẩn.")

        # Optimizer & Scheduler
        if self.cfg.optimizer.lower() == "sgd":
            self.optimizer = optim.SGD(
                self.model.parameters(),
                lr=self.cfg.lr0,
                momentum=self.cfg.momentum,
                weight_decay=self.cfg.weight_decay
            )
        else:
            self.optimizer = optim.AdamW(
                self.model.parameters(),
                lr=self.cfg.lr0,
                betas=self.cfg.betas,
                weight_decay=self.cfg.weight_decay
            )

        if self.cfg.use_scheduler:
            self.scheduler = optim.lr_scheduler.CosineAnnealingLR(
                self.optimizer,
                T_max=self.cfg.epochs,
                eta_min=self.cfg.lr0 * self.cfg.lr_min_factor
            )
        else:
            self.scheduler = None

        # Thư mục TensorBoard và Checkpoint
        experiment_name = getattr(self.cfg, "experiment_name", "drowsy_classifier_end2end")
        log_dir = Path(self.cfg.tb_log_dir) / experiment_name
        log_dir.mkdir(parents=True, exist_ok=True)
        self.writer = SummaryWriter(log_dir=str(log_dir))
        print(f"[*] TensorBoard log_dir: {log_dir}")

        self.ckpt_dir = Path(self.cfg.checkpoint_dir) / experiment_name
        self.ckpt_dir.mkdir(parents=True, exist_ok=True)

        self.scaler = torch.amp.GradScaler("cuda") if (self.cfg.amp and self.device.type == "cuda") else None

        if getattr(self.cfg, "early_stopping", False):
            self.early_stopper = EarlyStopping(
                patience=getattr(self.cfg, "patience", 10),
                min_delta=getattr(self.cfg, "min_delta", 1e-4),
                mode=getattr(self.cfg, "monitor_mode", "max"),
                monitor=getattr(self.cfg, "monitor_metric", "val_f1")
            )
        else:
            self.early_stopper = None

        history_path = getattr(self.cfg, "history_csv_path", None)
        self.history_csv_path = Path(history_path) if history_path else self.ckpt_dir / "training_history.csv"
        self._init_history_csv()

        self.global_step = 0
        self.best_val_f1 = 0.0
        self.start_epoch = 1

    def _init_history_csv(self) -> None:
        self.history_csv_path.parent.mkdir(parents=True, exist_ok=True)
        if not self.history_csv_path.exists():
            with open(self.history_csv_path, mode="w", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                writer.writerow([
                    "epoch", "train_loss", "train_acc", "train_recall", "train_f1",
                    "val_loss", "val_acc", "val_recall", "val_f1", "lr", "time_sec"
                ])

    def _log_history_csv(self, epoch: int, train_m: Dict[str, float], val_m: Optional[Dict[str, float]], elapsed_sec: float):
        lr = self.optimizer.param_groups[0]['lr']
        val_loss = f"{val_m['loss']:.4f}" if val_m else ""
        val_acc = f"{val_m['acc']:.4f}" if val_m else ""
        val_rec = f"{val_m['recall']:.4f}" if val_m else ""
        val_f1 = f"{val_m['f1']:.4f}" if val_m else ""

        with open(self.history_csv_path, mode="a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow([
                epoch,
                f"{train_m['loss']:.4f}",
                f"{train_m['acc']:.4f}",
                f"{train_m['recall']:.4f}",
                f"{train_m['f1']:.4f}",
                val_loss,
                val_acc,
                val_rec,
                val_f1,
                f"{lr:.6e}",
                f"{elapsed_sec:.2f}"
            ])

    def compute_loss_and_preds(self, logits: torch.Tensor, labels: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        if self.is_binary_bce:
            logits_1d = logits.view(-1)
            loss = self.criterion(logits_1d, labels.float().view(-1))
            preds = (torch.sigmoid(logits_1d) >= 0.5).long()
        else:
            loss = self.criterion(logits, labels.long())
            preds = torch.argmax(logits, dim=1)
        return loss, preds

    def train_epoch(self, epoch: int) -> Dict[str, float]:
        self.model.train()
        total_loss = 0.0
        all_preds = []
        all_targets = []
        valid_batches_count = 0

        accum_steps = max(1, getattr(self.cfg, "gradient_accumulation_steps", 1))
        accum_count = 0
        self.optimizer.zero_grad()

        num_batches = len(self.train_loader)
        pbar = tqdm(self.train_loader, desc=f"Train Epoch {epoch}/{self.cfg.epochs}", disable=not self.cfg.use_tqdm)

        for batch_idx, batch in enumerate(pbar):
            frames_t, labels, seq_lens, metas = batch
            if frames_t.size(0) == 0:
                continue

            frames_t = frames_t.to(self.device)
            labels = labels.to(self.device)
            seq_lens = seq_lens.to(self.device)

            try:
                autocast_enabled = (self.scaler is not None and self.device.type == "cuda")
                with torch.amp.autocast(device_type=self.device.type, enabled=autocast_enabled):
                    logits = self.model(frames_t, seq_lens=seq_lens)
                    loss, preds = self.compute_loss_and_preds(logits, labels)
                    loss_scaled = loss / accum_steps

                accum_count += 1
                if self.scaler is not None:
                    self.scaler.scale(loss_scaled).backward()
                else:
                    loss_scaled.backward()

                is_step_boundary = (accum_count == accum_steps) or ((batch_idx + 1) == num_batches)
                if is_step_boundary:
                    if self.scaler is not None:
                        self.scaler.unscale_(self.optimizer)
                        torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip_norm)
                        self.scaler.step(self.optimizer)
                        self.scaler.update()
                    else:
                        torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip_norm)
                        self.optimizer.step()
                    self.optimizer.zero_grad()
                    accum_count = 0

                unscaled_loss_item = loss.item()
                total_loss += unscaled_loss_item
                valid_batches_count += 1

                preds_np = preds.detach().cpu().numpy()
                targets_np = labels.detach().cpu().numpy()
                all_preds.extend(preds_np)
                all_targets.extend(targets_np)

                self.global_step += 1
                if self.cfg.enable_step_logging and self.global_step % self.cfg.log_step_interval == 0:
                    step_metrics = calculate_metrics(preds_np, targets_np)
                    self.writer.add_scalar("Train_Step/Loss", unscaled_loss_item, self.global_step)
                    self.writer.add_scalar("Train_Step/Accuracy", step_metrics["acc"], self.global_step)
                    self.writer.add_scalar("Train_Step/Recall", step_metrics["recall"], self.global_step)
                    self.writer.add_scalar("Train_Step/F1", step_metrics["f1"], self.global_step)

                pbar.set_postfix({"loss": f"{unscaled_loss_item:.4f}"})

            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print(f"\n[CẢNH BÁO] CUDA Out of Memory tại batch {batch_idx}. Đang dọn dẹp bộ nhớ và bỏ qua batch...")
                    self.optimizer.zero_grad()
                    accum_count = 0
                    try:
                        del frames_t, labels, seq_lens, metas
                    except Exception:
                        pass
                    del e
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    continue
                else:
                    raise e

        avg_loss = total_loss / max(valid_batches_count, 1)
        epoch_metrics = calculate_metrics(np.array(all_preds), np.array(all_targets))
        epoch_metrics["loss"] = avg_loss
        return epoch_metrics

    @torch.no_grad()
    def validate_epoch(self, epoch: int) -> Dict[str, float]:
        self.model.eval()
        total_loss = 0.0
        all_preds = []
        all_targets = []
        valid_batches_count = 0

        pbar = tqdm(self.val_loader, desc=f"Val Epoch {epoch}/{self.cfg.epochs}", disable=not self.cfg.use_tqdm)
        for batch_idx, batch in enumerate(pbar):
            frames_t, labels, seq_lens, metas = batch
            if frames_t.size(0) == 0:
                continue

            frames_t = frames_t.to(self.device)
            labels = labels.to(self.device)
            seq_lens = seq_lens.to(self.device)

            try:
                autocast_enabled = (self.cfg.amp and self.device.type == "cuda")
                with torch.amp.autocast(device_type=self.device.type, enabled=autocast_enabled):
                    logits = self.model(frames_t, seq_lens=seq_lens)
                    loss, preds = self.compute_loss_and_preds(logits, labels)

                loss_item = loss.item()
                total_loss += loss_item
                valid_batches_count += 1

                preds_np = preds.cpu().numpy()
                targets_np = labels.cpu().numpy()
                all_preds.extend(preds_np)
                all_targets.extend(targets_np)

            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print(f"\n[CẢNH BÁO] CUDA Out of Memory tại validation batch {batch_idx}. Đang bỏ qua batch...")
                    try:
                        del frames_t, labels, seq_lens, metas
                    except Exception:
                        pass
                    del e
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    continue
                else:
                    raise e

        avg_loss = total_loss / max(valid_batches_count, 1)
        epoch_metrics = calculate_metrics(np.array(all_preds), np.array(all_targets))
        epoch_metrics["loss"] = avg_loss
        return epoch_metrics

    def _resolve_checkpoint_path(self, checkpoint_path: str) -> Path:
        alias = str(checkpoint_path).strip().lower()
        if alias in ("last", "last.pt"):
            return self.ckpt_dir / "last.pt"
        elif alias in ("best", "best.pt"):
            return self.ckpt_dir / "best.pt"
        target = Path(checkpoint_path)
        if not target.is_absolute():
            candidate = self.ckpt_dir / target
            if candidate.exists():
                return candidate
        return target

    def load_checkpoint(self, checkpoint_path: str) -> None:
        target_path = self._resolve_checkpoint_path(checkpoint_path)
        print(f"[*] Đang khôi phục từ checkpoint: {target_path}")
        if not target_path.exists():
            raise FileNotFoundError(f"Không tìm thấy checkpoint tại: {target_path}")

        ckpt = torch.load(target_path, map_location=self.device)
        self.model.load_state_dict(ckpt["model_state_dict"])
        self.optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        self.start_epoch = ckpt.get("epoch", 0) + 1
        self.best_val_f1 = ckpt.get("best_val_f1", 0.0)
        self.global_step = ckpt.get("global_step", (self.start_epoch - 1) * len(self.train_loader))

        if self.scheduler is not None and "scheduler_state_dict" in ckpt:
            self.scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        if self.scaler is not None and "scaler_state_dict" in ckpt:
            self.scaler.load_state_dict(ckpt["scaler_state_dict"])
            print("[*] Đã khôi phục thành công trạng thái GradScaler (AMP).")
        print(f"[*] Đã khôi phục thành công. Tiếp tục từ epoch {self.start_epoch} (Best F1 trước: {self.best_val_f1:.4f})")

    def _atomic_save(self, state: Dict[str, Any], target_path: Path) -> None:
        tmp_path = target_path.with_suffix(f"{target_path.suffix}.tmp")
        torch.save(state, tmp_path)
        if target_path.exists():
            target_path.unlink()
        tmp_path.rename(target_path)

    def save_checkpoint(self, epoch: int, is_best: bool = False, val_f1: Optional[float] = None) -> None:
        if is_best and val_f1 is not None and val_f1 > self.best_val_f1:
            self.best_val_f1 = val_f1

        state = {
            "epoch": epoch,
            "global_step": self.global_step,
            "model_state_dict": self.model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "best_val_f1": self.best_val_f1,
            "val_f1": val_f1,
            "config": self.cfg.to_dict()
        }
        if self.scheduler is not None:
            state["scheduler_state_dict"] = self.scheduler.state_dict()
        if self.scaler is not None:
            state["scaler_state_dict"] = self.scaler.state_dict()

        last_path = self.ckpt_dir / "last.pt"
        self._atomic_save(state, last_path)

        if is_best:
            best_path = self.ckpt_dir / "best.pt"
            self._atomic_save(state, best_path)
            print(f"[*] Đã lưu mô hình tốt nhất (best.pt) tại epoch {epoch} với F1: {self.best_val_f1:.4f}")

        save_all = getattr(self.cfg, "save_all_epochs", False)
        save_interval = getattr(self.cfg, "save_ckpt_interval_epochs", 1)
        if save_all or (save_interval > 0 and epoch % save_interval == 0):
            epoch_path = self.ckpt_dir / f"epoch_{epoch}.pt"
            self._atomic_save(state, epoch_path)

        keep_last = getattr(self.cfg, "ckpt_keep_last", None)
        if keep_last and keep_last > 0:
            epoch_files = sorted(
                self.ckpt_dir.glob("epoch_*.pt"),
                key=lambda p: int(p.stem.split("_")[1]) if p.stem.split("_")[1].isdigit() else 0
            )
            if len(epoch_files) > keep_last:
                for old_f in epoch_files[:-keep_last]:
                    try:
                        old_f.unlink()
                    except Exception:
                        pass

    def train(self) -> None:
        print("[*] Bắt đầu quá trình huấn luyện End-to-End DrowsyClassifier...")
        for epoch in range(self.start_epoch, self.cfg.epochs + 1):
            t_epoch_start = time.time()
            train_metrics = self.train_epoch(epoch)

            self.writer.add_scalar("Train_Epoch/Loss", train_metrics["loss"], epoch)
            self.writer.add_scalar("Train_Epoch/Accuracy", train_metrics["acc"], epoch)
            self.writer.add_scalar("Train_Epoch/Recall", train_metrics["recall"], epoch)
            self.writer.add_scalar("Train_Epoch/F1", train_metrics["f1"], epoch)
            self.writer.add_scalar("Train_Epoch/LearningRate", self.optimizer.param_groups[0]['lr'], epoch)

            run_val = (len(self.val_loader.dataset) > 0) and (
                    (epoch % self.cfg.val_interval_epochs == 0) or (epoch == self.cfg.epochs)
            )

            is_best = False
            val_metrics = None
            if run_val:
                val_metrics = self.validate_epoch(epoch)
                self.writer.add_scalar("Val_Epoch/Loss", val_metrics["loss"], epoch)
                self.writer.add_scalar("Val_Epoch/Accuracy", val_metrics["acc"], epoch)
                self.writer.add_scalar("Val_Epoch/Recall", val_metrics["recall"], epoch)
                self.writer.add_scalar("Val_Epoch/F1", val_metrics["f1"], epoch)

                print(f"Epoch {epoch:02d}/{self.cfg.epochs} | Train Loss: {train_metrics['loss']:.4f} | "
                      f"Val Loss: {val_metrics['loss']:.4f} | Val F1: {val_metrics['f1']:.4f} | Val Acc: {val_metrics['acc']:.4f}")

                if val_metrics["f1"] > self.best_val_f1:
                    self.best_val_f1 = val_metrics["f1"]
                    is_best = True
            else:
                print(f"Epoch {epoch:02d}/{self.cfg.epochs} | Train Loss: {train_metrics['loss']:.4f} | Train F1: {train_metrics['f1']:.4f} | (Val: Bỏ qua)")

            self.save_checkpoint(epoch=epoch, is_best=is_best, val_f1=val_metrics["f1"] if val_metrics else None)
            elapsed_sec = time.time() - t_epoch_start
            self._log_history_csv(epoch, train_metrics, val_metrics, elapsed_sec)

            if run_val and self.early_stopper is not None and val_metrics is not None:
                monitor_key = self.early_stopper.monitor.replace("val_", "")
                target_score = val_metrics.get(monitor_key, val_metrics.get(self.early_stopper.monitor, val_metrics["f1"]))
                if self.early_stopper.step(target_score, epoch):
                    print(f"[*] Early Stopping kích hoạt tại epoch {epoch}! (Kỷ nguyên tối ưu: {self.early_stopper.best_epoch})")
                    break

            if self.scheduler is not None:
                self.scheduler.step()

            if self.cfg.empty_cache_interval > 0 and epoch % self.cfg.empty_cache_interval == 0:
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        print("[*] Đã hoàn thành quá trình huấn luyện End-to-End DrowsyClassifier.")
        self.writer.close()

print("[✓] Đã khởi tạo hạ tầng huấn luyện Trainer thành công!")

## 🚀 Section 7: Thực thi Huấn luyện Tương tác
Khởi tạo Trainer, xác nhận cấu trúc mô hình End-to-End và bắt đầu quá trình huấn luyện mô hình.

In [ ]:
# 1. Khởi tạo đối tượng Trainer và kiểm tra cấu trúc mô hình
trainer = Trainer(cfg)

# Khôi phục nếu enable_resume = True hoặc resume được chỉ định
if cfg.enable_resume and cfg.resume:
    trainer.load_checkpoint(cfg.resume)

# 2. Bắt đầu quá trình huấn luyện mô hình!
# Quá trình này sẽ hiển thị thanh tiến trình trực quan tqdm qua từng epoch
trainer.train()

## 📊 Section 8: Trực quan hóa & Đánh giá mô hình Sau Huấn luyện
1. Vẽ đồ thị Loss, Accuracy, F1 và Learning Rate qua các epochs từ `training_history.csv`.
2. Đánh giá Ma trận Nhầm lẫn Confusion Matrix của mô hình tối ưu nhất (`best.pt`).

In [ ]:
# 1. Vẽ đồ thị Loss, Accuracy, F1 và Learning Rate qua các epochs
history_file = Path(cfg.history_csv_path)

if history_file.exists():
    epochs_list, train_loss, val_loss = [], [], []
    train_f1, val_f1, train_acc, val_acc, lr_list = [], [], [], [], []

    with open(history_file, mode="r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            epochs_list.append(int(row["epoch"]))
            train_loss.append(float(row["train_loss"]))
            val_loss.append(float(row["val_loss"]) if row["val_loss"] else float("nan"))
            train_f1.append(float(row["train_f1"]))
            val_f1.append(float(row["val_f1"]) if row["val_f1"] else float("nan"))
            train_acc.append(float(row["train_acc"]))
            val_acc.append(float(row["val_acc"]) if row["val_acc"] else float("nan"))
            lr_list.append(float(row["lr"]))

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    # Biểu đồ Loss
    axes[0, 0].plot(epochs_list, train_loss, 'o-', color='tab:blue', label='Train Loss')
    if not all(math.isnan(v) for v in val_loss):
        axes[0, 0].plot(epochs_list, val_loss, 's--', color='tab:red', label='Val Loss')
    axes[0, 0].set_title("Training & Validation Loss")
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].set_ylabel("Loss")
    axes[0, 0].grid(True, linestyle="--", alpha=0.6)
    axes[0, 0].legend()

    # Biểu đồ F1-Score
    axes[0, 1].plot(epochs_list, train_f1, 'o-', color='tab:green', label='Train F1')
    if not all(math.isnan(v) for v in val_f1):
        axes[0, 1].plot(epochs_list, val_f1, 's--', color='tab:orange', label='Val F1')
    axes[0, 1].set_title("Macro F1-Score")
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].set_ylabel("F1 Score")
    axes[0, 1].grid(True, linestyle="--", alpha=0.6)
    axes[0, 1].legend()

    # Biểu đồ Accuracy
    axes[1, 0].plot(epochs_list, train_acc, 'o-', color='tab:purple', label='Train Acc')
    if not all(math.isnan(v) for v in val_acc):
        axes[1, 0].plot(epochs_list, val_acc, 's--', color='tab:pink', label='Val Acc')
    axes[1, 0].set_title("Accuracy")
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].set_ylabel("Accuracy")
    axes[1, 0].grid(True, linestyle="--", alpha=0.6)
    axes[1, 0].legend()

    # Biểu đồ Learning Rate
    axes[1, 1].plot(epochs_list, lr_list, 'd-', color='tab:brown', label='Learning Rate')
    axes[1, 1].set_title("Learning Rate Schedule")
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].set_ylabel("LR")
    axes[1, 1].set_yscale("log")
    axes[1, 1].grid(True, linestyle="--", alpha=0.6)
    axes[1, 1].legend()

    plt.tight_layout()
    plt.savefig(Path(cfg.checkpoint_dir) / f"{cfg.experiment_name}_learning_curves.png", dpi=300)
    plt.show()
else:
    print(f"[!] Không tìm thấy file lịch sử {history_file}")

In [ ]:
# 2. Đánh giá Ma trận Nhầm lẫn Confusion Matrix của mô hình tối ưu nhất (best.pt)
best_ckpt_path = Path(cfg.checkpoint_dir) / cfg.experiment_name / "epoch_2.pt"
if not best_ckpt_path.exists():
    best_ckpt_path = Path(cfg.checkpoint_dir) / "best.pt"

if best_ckpt_path.exists():
    print(f"[*] Đang nạp mô hình tối ưu nhất từ {best_ckpt_path} để đánh giá Validation...")
    ckpt = torch.load(best_ckpt_path, map_location=trainer.device)
    trainer.model.load_state_dict(ckpt["model_state_dict"])
    trainer.model.eval()

    all_val_preds, all_val_targets = [], []
    with torch.no_grad():
        for batch in tqdm(trainer.val_loader, desc="Đánh giá best.pt"):
            frames_t, labels, seq_lens, metas = batch
            if frames_t.size(0) == 0:
                continue
            frames_t = frames_t.to(trainer.device)
            labels = labels.to(trainer.device)
            seq_lens = seq_lens.to(trainer.device)

            logits = trainer.model(frames_t, seq_lens=seq_lens)
            _, preds = trainer.compute_loss_and_preds(logits, labels)

            all_val_preds.extend(preds.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    cm = confusion_matrix(all_val_targets, all_val_preds)
    cr = classification_report(all_val_targets, all_val_preds, target_names=["Alert (0)", "Drowsy (1)"], digits=4)

    print("\n" + "=" * 50)
    print("           BÁO CÁO PHÂN LOẠI CHI TIẾT (BEST MODEL)")
    print("=" * 50)
    print(cr)

    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=["Alert (0)", "Drowsy (1)"],
                yticklabels=["Alert (0)", "Drowsy (1)"])
    plt.title(f"Ma trận Nhầm lẫn Confusion Matrix ({cfg.experiment_name})")
    plt.xlabel("Nhãn Dự đoán (Predicted)")
    plt.ylabel("Nhãn Thực tế (Actual)")
    plt.tight_layout()
    plt.savefig(Path(cfg.checkpoint_dir) / f"{cfg.experiment_name}_confusion_matrix.png", dpi=300)
    plt.show()
else:
    print(f"[!] Không tìm thấy checkpoint best.pt tại {best_ckpt_path}")

## 📦 Section 9: Đóng gói Checkpoints & Tải về Máy tính (Export Artifacts)
Tự động gom toàn bộ các tệp quan trọng (`best.pt`, `last.pt`, `training_history.csv`, biểu đồ) thành các file ZIP nén gọn để tải trực tiếp từ giao diện Kaggle Output.

In [ ]:
# 1. Đóng gói thư mục logs và checkpoints thành file zip
!zip -r logs.zip ./logs
!zip -r checkpoints.zip ./checkpoints
print("[✓] Đã tạo thành công logs.zip và checkpoints.zip trong thư mục làm việc!")